<h1 style="color: purple;">Education & Digital Access: Data Analysis</h1>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import statsmodels.api as sm
import math
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.decomposition import PCA
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GroupShuffleSplit

<h2 style="color: orange;">Section 1: Data Cleaning</h2>

This sections contains all the steps that I did to clean the row data I have. It contains 4 sub-sections for the 4 differnet data sets that I gathered about education and digital access indices.

<h3 style="color: teal;">Data 1: Tertiary Education per Country</h2>

In [ ]:
file_path = "../data/API_SE.TER.ENRR_DS2_en_excel_v2_802.xls"   
df = pd.read_excel(file_path, engine="xlrd")

df.columns = df.columns.astype(str).str.strip()
year_cols = [c for c in df.columns if c.isdigit()]

In [ ]:
df_long = df.melt(id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
    value_vars=year_cols,var_name="Year",value_name="Tertiary_Enrollment_Gross")

df_long["Year"] = df_long["Year"].astype(int)
df_long["Tertiary_Enrollment_Gross"] = pd.to_numeric(df_long["Tertiary_Enrollment_Gross"])

df_long = df_long.dropna(subset=["Tertiary_Enrollment_Gross"]).copy()
df_clean = df_long.drop(columns=["Indicator Name", "Indicator Code"]).copy()

print(df_clean.head())

In [ ]:
print(df_clean.info())

In [ ]:
tertiary_enrollment_data  = df_clean[df_clean["Year"].between(2015, 2019)].copy()
print(tertiary_enrollment_data .shape)

In [ ]:
tertiary_enrollment_data.head(10)

<h3 style="color: teal;">Data 2: Mobile Connectivity Index</h2>

In [ ]:
file_path = "../data/MCI_data_2025.xlsx"

mci_data = pd.read_excel(file_path, sheet_name="Index Scores")

mci_data.columns = mci_data.columns.str.strip()

for col in ["ISO Code", "Country", "Region", "Cluster"]:
    if col in mci_data.columns:
        mci_data[col] = mci_data[col].astype(str).str.strip()

mci_data["Year"] = pd.to_numeric(mci_data["Year"])

for col in mci_data.columns:
    if col not in ["ISO Code", "Country", "Region", "Cluster"]:
        mci_data[col] = pd.to_numeric(mci_data[col],)

mci_data = mci_data[mci_data["Year"].between(2015, 2019)].copy()
mci_data.info()

In [ ]:
mci_data.head(5)

<h3 style="color: teal;">Data 3: Percentage of People Using Internet</h2>

In [ ]:
file_path = "../data/share-of-individuals-using-the-internet.csv"

internet_data = pd.read_csv(file_path)
internet_data.head()

In [ ]:
internet_data.info()

In [ ]:
internet_data.columns = internet_data.columns.str.strip()

internet_data = internet_data.rename(columns={"Entity": "Country","Code": "Country Code",
    "Year": "Year","Share of the population using the Internet": "Internet_Use_Share"})

for col in ["Country", "Country Code"]:
    internet_data[col] = internet_data[col].astype(str).str.strip()

internet_data["Year"] = pd.to_numeric(internet_data["Year"], errors="coerce")
internet_data["Internet_Use_Share"] = pd.to_numeric(internet_data["Internet_Use_Share"], errors="coerce")

internet_data = internet_data[internet_data["Year"].between(2015, 2019)].copy()
internet_data.head()

In [ ]:
internet_data.isna().sum()

<h3 style="color: teal;">Data 4: World Education Data</h2>

In [ ]:
file_path = "../data/world-education-data.csv"
education_data = pd.read_csv(file_path)
education_data.head()

In [ ]:
education_data.info()

In [ ]:
education_data.columns = education_data.columns.str.strip()

for col in ["country", "country_code"]:
    education_data[col] = education_data[col].astype(str).str.strip()

education_data["year"] = pd.to_numeric(education_data["year"], errors="coerce")

for col in education_data.columns:
    if col not in ["country", "country_code", "year"]:
        education_data[col] = pd.to_numeric(education_data[col], errors="coerce")

education_data = education_data[education_data["year"].between(2015, 2019)].copy()
education_data.head()

In [ ]:
education_data.isna().sum()

In [ ]:
#I like to save data as scv and re-open via Excel just to doable check that everything is fine.
tertiary_enrollment_data.to_csv("../data/clean_data_1_tertiary_education_per_country.csv", index=False)
mci_data.to_csv("../data/clean_data_2_mobile_connectivity_index.csv", index=False)
internet_data.to_csv("../data/clean_data_3_percentage_of_people_using_internet.csv", index=False)
education_data.to_csv("../data/clean_data_4_world_education_data.csv", index=False)

<h2 style="color: orange;">Section 2: Data Merging</h2>

In this section, I start by merging the 4 datasets that I cleaned in Section 1 all together using the key ID of country name and code. Then, I merge it with the happiness data, which contains the response variable of my research happiness_score. It seems long because the happiness data is divided into $5$ different csv files for each year from $2015$ to $2019$.

In [ ]:
data1 = pd.read_csv("../data/clean_data_1_tertiary_education_per_country.csv")
data2 = pd.read_csv("../data/clean_data_2_mobile_connectivity_index.csv")
data3 = pd.read_csv("../data/clean_data_3_percentage_of_people_using_internet.csv")
data4 = pd.read_csv("../data/clean_data_4_world_education_data.csv")

In [ ]:
#to match names
data1 = data1.rename(columns={"Country Name": "country", "Country Code": "country_code", "Year": "year"})
data2 = data2.rename(columns={"Country": "country", "ISO Code": "country_code", "Year": "year"})
data3 = data3.rename(columns={"Country": "country", "Country Code": "country_code", "Year": "year"})

In [ ]:
print("Duplicates in data1:", data1.duplicated(subset=["country_code", "year"]).sum())
print("Duplicates in data2:", data2.duplicated(subset=["country_code", "year"]).sum())
print("Duplicates in data3:", data3.duplicated(subset=["country_code", "year"]).sum())
print("Duplicates in data4:", data4.duplicated(subset=["country_code", "year"]).sum())

In [ ]:
merged_data = data1.merge(data2.drop(columns=["country"], errors="ignore"),on=["country_code", "year"],how="outer")

merged_data = merged_data.merge(data3.drop(columns=["country"], errors="ignore"),on=["country_code", "year"],how="outer")

merged_data = merged_data.merge(data4.drop(columns=["country"], errors="ignore"),on=["country_code", "year"],how="outer")

mapping = pd.concat([data1[["country", "country_code"]],data2[["country", "country_code"]],
    data3[["country", "country_code"]],data4[["country", "country_code"]]], ignore_index=True)

mapping = mapping.dropna().drop_duplicates()

In [ ]:
code_to_country = mapping.drop_duplicates(subset=["country_code"]).set_index("country_code")["country"]

merged_data["country"] = merged_data["country"].fillna(merged_data["country_code"].map(code_to_country))

merged_data = merged_data.dropna(subset=["country", "country_code", "year"]).copy()

In [ ]:
front_cols = ["country", "country_code", "year"]
other_cols = [col for col in merged_data.columns if col not in front_cols]
merged_data = merged_data[front_cols + other_cols]

In [ ]:
merged_data.to_csv("../data/merged_education_digital_data_final.csv", index=False)
merged_data.to_excel("../data/merged_education_digital_data_final.xlsx", index=False)

In [ ]:
h2015 = pd.read_csv("../data/2015.csv")
h2016 = pd.read_csv("../data/2016.csv")
h2017 = pd.read_csv("../data/2017.csv")
h2018 = pd.read_csv("../data/2018.csv")
h2019 = pd.read_csv("../data/2019.csv")

h2015 = h2015[["Country", "Happiness Rank", "Happiness Score"]].rename(
    columns={"Country": "country", "Happiness Rank": "happiness_rank", "Happiness Score": "happiness_score"})
h2015["year"] = 2015
h2016 = h2016[["Country", "Happiness Rank", "Happiness Score"]].rename(
    columns={"Country": "country", "Happiness Rank": "happiness_rank", "Happiness Score": "happiness_score"})
h2016["year"] = 2016
h2017 = h2017[["Country", "Happiness.Rank", "Happiness.Score"]].rename(
    columns={"Country": "country", "Happiness.Rank": "happiness_rank", "Happiness.Score": "happiness_score"})
h2017["year"] = 2017
h2018 = h2018[["Country or region", "Overall rank", "Score"]].rename(
    columns={"Country or region": "country", "Overall rank": "happiness_rank", "Score": "happiness_score"})
h2018["year"] = 2018
h2019 = h2019[["Country or region", "Overall rank", "Score"]].rename(
    columns={"Country or region": "country", "Overall rank": "happiness_rank", "Score": "happiness_score"})
h2019["year"] = 2019

happiness_data = pd.concat([h2015, h2016, h2017, h2018, h2019], ignore_index=True)

In [ ]:
happiness_data["country"] = happiness_data["country"].astype(str).str.strip()
happiness_data["year"] = pd.to_numeric(happiness_data["year"], errors="coerce")
happiness_data["happiness_rank"] = pd.to_numeric(happiness_data["happiness_rank"], errors="coerce")
happiness_data["happiness_score"] = pd.to_numeric(happiness_data["happiness_score"], errors="coerce")

In [ ]:
name_map = {"Czech Republic": "Czechia",
    "Turkey": "Turkiye",
    "Russia": "Russian Federation",
    "Vietnam": "Viet Nam",
    "Egypt": "Egypt, Arab Rep.",
    "Iran": "Iran, Islamic Rep.",
    "Hong Kong": "Hong Kong SAR, China",
    "Hong Kong S.A.R., China": "Hong Kong SAR, China",
    "South Korea": "Korea, Rep.",
    "Kyrgyzstan": "Kyrgyz Republic",
    "Laos": "Lao PDR",
    "Ivory Coast": "Cote d'Ivoire",
    "Macedonia": "North Macedonia",
    "Swaziland": "Eswatini",
    "Syria": "Syrian Arab Republic",
    "Taiwan Province of China": "Taiwan",
    "Trinidad & Tobago": "Trinidad and Tobago",
    "Congo (Brazzaville)": "Congo, Rep.",
    "Congo (Kinshasa)": "Congo, Dem. Rep.",
    "Palestinian Territories": "West Bank and Gaza",
    "Northern Cyprus": "North Cyprus"}

In [ ]:
merged_data["country_merge"] = merged_data["country"].replace(name_map)
happiness_data["country_merge"] = happiness_data["country"].replace(name_map)

In [ ]:
final_data = merged_data.merge(happiness_data[["country_merge", "year", "happiness_rank", "happiness_score"]],
    on=["country_merge", "year"],how="left")

In [ ]:
final_data = final_data.drop(columns=["country_merge"])
final_data = final_data.dropna(subset=["happiness_rank", "happiness_score"]).copy()

In [ ]:
print(final_data.shape)

In [ ]:
print(final_data[["country", "country_code", "year", "happiness_rank", "happiness_score"]].isna().sum())

In [ ]:
final_data.to_csv("../data/merged_education_digital_happiness_data_no_missing.csv", index=False)
final_data.to_excel("../data/merged_education_digital_happiness_data_no_missing.xlsx", index=False)

In [ ]:
final_data.head()

<h2 style="color: orange;">Section 3: Exploratory Data Analysis</h2>

In this section, I do the basic EDA divided into sub-sections to be easy to go through.

<h3 style="color: teal;">3.1 General EDA</h2>

In [ ]:
data_fr = pd.read_csv("../data/merged_education_digital_happiness_data_no_missing.csv")

In [ ]:
data_fr.tail(3)

In [ ]:
print(data_fr.dtypes.to_string())

In [ ]:
display(data_fr.describe(include='all').T)

In [ ]:
#dropping na data points to easily do the eda
data_fr = data_fr.dropna(axis=1, how="all")
data_fr = data_fr.loc[:, data_fr.nunique(dropna=True) > 1]

# I doped these columns because I found they are repeated.
data_fr = data_fr.drop(columns=["Mobile ownership","Cybersecurity Index",
    "school_enrol_tertiary_pct"], errors="ignore")

In [ ]:
num_cols = data_fr.select_dtypes(include=np.number).columns.tolist()
cat_cols = data_fr.select_dtypes(exclude=np.number).columns.tolist()

In [ ]:
print("Numeric columns:", len(num_cols))
print(num_cols)

In [ ]:
print("Categorical columns:", len(cat_cols))
print(cat_cols)

In [ ]:
print("Duplicate full rows:", data_fr.duplicated().sum())

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=data_fr, x="Region", order=data_fr["Region"].value_counts().index)
plt.title("Count of Data by Region")
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=data_fr, x="Cluster", order=data_fr["Cluster"].value_counts().index)
plt.title("Count of Data by Cluster")
plt.xticks(rotation=0)
plt.show()

In [ ]:
region_happiness = data_fr.groupby("Region")["happiness_score"].mean().sort_values(ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x=region_happiness.index, y=region_happiness.values)
plt.title("Average Happiness Score by Region")
plt.xticks(rotation=45)
plt.ylabel("Average happiness_score")
plt.show()

In [ ]:
cluster_happiness = data_fr.groupby("Cluster")["happiness_score"].mean().sort_values(ascending=False)
plt.figure(figsize=(10, 6))
sns.barplot(x=cluster_happiness.index, y=cluster_happiness.values)
plt.title("Average Happiness Score by Cluster")
plt.xticks(rotation=45)
plt.ylabel("Average happiness_score")
plt.show()

In [ ]:
top_countries = data_fr.groupby("country")["happiness_score"].mean().sort_values(ascending=False).head(30)
plt.figure(figsize=(14, 10))
sns.barplot(x=top_countries.values, y=top_countries.index)
plt.title("Top 30 Countries by Average Happiness Score")
plt.xlabel("Average happiness_score")
plt.grid(True)
plt.ylabel("country")
plt.show()

In [ ]:
cols = num_cols
rows = math.ceil(len(cols) / 5)

plt.figure(figsize=(25, 5 * rows))

for i, col in enumerate(cols):
    plt.subplot(rows, 5, i+1)
    sns.kdeplot(data=data_fr, x=col, fill=True)
    plt.title(col, fontsize=9)
plt.tight_layout()
plt.show()

We see that the numerical variables follow differnet distributions, some can be modelled as normal distributions with skewness and other can be modelled as bimodal as Digital Language Support variable.

In [ ]:
year_avg = data_fr.groupby("year")[["happiness_score", "Internet_Use_Share", "Index", "Tertiary_Enrollment_Gross"]].mean().reset_index()
plt.figure(figsize=(10, 6))
sns.lineplot(data=year_avg, x="year", y="happiness_score", marker="o")
plt.title("Average Happiness Score Over Time")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=year_avg, x="year", y="Index", marker="o")
plt.title("Average Digital Index Over Time")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=year_avg, x="year", y="Tertiary_Enrollment_Gross", marker="o")
plt.title("Average Tertiary Enrollment Gross Over Time")
plt.show()

<h3 style="color: teal;">3.2 Correlation Study</h2>

This subsection provide a brief study about correlation using correlation matrix and VIF values.

In [ ]:
corr_data = data_fr[num_cols].copy()
corr_matrix = corr_data.corr()

In [ ]:
happiness_corr_table = corr_matrix[["happiness_score"]].drop("happiness_score")

happiness_corr_table = happiness_corr_table.rename(columns={"happiness_score": "correlation_with_happiness_score"})

happiness_corr_table["absolute_correlation"] = happiness_corr_table["correlation_with_happiness_score"].abs()

happiness_corr_table = happiness_corr_table.sort_values("absolute_correlation", ascending=False)
display(happiness_corr_table)

In [ ]:
correlationss = happiness_corr_table.drop(index="happiness_rank", errors="ignore")

correlationss = correlationss.sort_values("correlation_with_happiness_score")

plt.figure(figsize=(18, 16))
sns.barplot(x=correlationss["correlation_with_happiness_score"],y=correlationss.index)
plt.title("Correlation of Numerical Variables with Happiness Score")
plt.xlabel("Correlation with Happiness Score")
plt.ylabel("Variable")
plt.grid(axis="x")
plt.show()

In [ ]:
# here I dropped overlaping variables because originally I got infinite values of VIF .
drop_vif_cols = ["happiness_score","Index","Infrastructure","Affordability","Consumer Readiness",
    "Content and Services","Network coverage","Network performance","Spectrum","Mobile data affordability",
    "Handset affordability","Taxation","Mobile Ownership","Basic Skills","Gender Equality","Local Relevance",
    "Online Security","Mobile ownership","Cybersecurity Index","happiness_rank"]

vif_cols = [col for col in num_cols if col not in drop_vif_cols]

vif_data = data_fr[vif_cols].copy()

vif_data = vif_data.dropna(axis=1, how="all")
vif_data = vif_data.loc[:, vif_data.nunique(dropna=True) > 1]

vif_data = vif_data.fillna(vif_data.median())

X = sm.add_constant(vif_data)

vif_results = []

for i in range(1, X.shape[1]):
    vif_results.append([X.columns[i], variance_inflation_factor(X.values, i)])

vif_table = pd.DataFrame(vif_results, columns=["variable", "VIF"])
vif_table = vif_table.sort_values("VIF", ascending=False)

In [ ]:
vifffss = vif_table.copy()
vifffss = vifffss.sort_values("VIF")

plt.figure(figsize=(10, 8))
sns.barplot(x=vifffss["VIF"], y=vifffss["variable"])
plt.title("VIF Values")
plt.xlabel("VIF")
plt.ylabel("Variable")
plt.grid(True)
plt.show()

<h3 style="color: teal;">3.3 Missing Values Analysis</h2>

This sub section provides a brief analysis of missing data.

In [ ]:
print("Total missing values:", data_fr.isna().sum().sum())

In [ ]:
missing_counts = data_fr.isnull().sum().sort_values(ascending=True)
missing_counts = missing_counts[missing_counts > 0]

plt.figure(figsize=(20,18))
plt.barh(missing_counts.index, missing_counts.values)
plt.title("Missing Values Count by Predictor",size = 16)
plt.xlabel("Number of Missing Values")
plt.grid(axis="x", alpha=0.3)
plt.xticks(np.arange(0, missing_counts.values.max() + 20, 20))
plt.ylabel("predictor")
plt.show()

In [ ]:
missing_pct = (data_fr.isnull().sum() / len(data_fr)) * 100
missing_pct = missing_pct[missing_pct > 5].sort_values(ascending=False)

labels = missing_pct.index

plt.figure(figsize=(15, 15))
plt.pie(missing_pct.values, labels=labels, autopct='%1.2f%%', startangle=90)
plt.title('predictors with Missing Values Percentage > 5.0%')
plt.show()

In [ ]:
missing_by_year = data_fr.groupby("year").apply(lambda x: x.isnull().sum().sum(),include_groups=False)

plt.figure(figsize=(8,5))
plt.bar(missing_by_year.index, missing_by_year.values)

plt.title("Total Missing Values by Year")
plt.xlabel("Year")
plt.ylabel("Number of Missing Values")
plt.show()

In [ ]:
missing_per_row = data_fr.isnull().sum(axis=1)

plt.figure(figsize=(10,8))

counts, bins, patches = plt.hist(missing_per_row, bins=40)

plt.title("Number of Missing Values per Row")
plt.xlabel("Missing Values in One Row")
plt.ylabel("Number of Rows")

plt.xticks(np.arange(0, missing_per_row.max() + 2, 2))

# y-axis ticks by 10
plt.yticks(np.arange(0, counts.max() + 10, 10))

plt.grid(axis="y", alpha=0.3)
plt.show()

There are few rows that have an extreme number of missing data! this should be taken into consideration

<h3 style="color: teal;">3.4 Outliers Analysis</h2>

In [ ]:
remove_cols = ["year", "happiness_rank"]
num_cols_clean = [col for col in num_cols if col not in remove_cols]

outlier_summary = []
for col in num_cols_clean:
    q1 = data_fr[col].quantile(0.25)
    q3 = data_fr[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    
    outlier_count = ((data_fr[col] < lower) | (data_fr[col] > upper)).sum()
    outlier_pct = (outlier_count / len(data_fr)) * 100
    outlier_summary.append([col, outlier_count, round(outlier_pct, 2)])

outlier_summary = pd.DataFrame(outlier_summary,columns=["variable", "outlier_count", "outlier_percent"])
outlier_summary = outlier_summary[outlier_summary["outlier_count"] > 0]
display(outlier_summary.sort_values("outlier_count", ascending=False).reset_index())

In [ ]:
outlier_vars = outlier_summary["variable"].tolist()

In [ ]:
plot_data = outlier_summary.sort_values("outlier_count", ascending=True)

plt.figure(figsize=(12,8))
plt.barh(plot_data["variable"], plot_data["outlier_count"])

plt.title("Outlier Count by Variable")
plt.xlabel("Number of Outliers")
plt.ylabel("Variable")

plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,8))
plt.barh(plot_data["variable"], plot_data["outlier_percent"])

plt.title("Outlier Percentage by Variable")
plt.xlabel("Outlier Percentage (%)")
plt.ylabel("Variable")

plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
standardized_outlier_data = (data_fr[outlier_vars] - data_fr[outlier_vars].mean()) / data_fr[outlier_vars].std()

plt.figure(figsize=(12,10))
standardized_outlier_data.boxplot(vert=False)

plt.title("Standardized Boxplots for Variables with Outliers")
plt.ylabel("Standardized Value")
plt.xticks(rotation=0)

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
outlier_flags = pd.DataFrame(index=data_fr.index)

for col in outlier_vars:
    q1 = data_fr[col].quantile(0.25)
    q3 = data_fr[col].quantile(0.75)
    iqr = q3 - q1
    
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    
    outlier_flags[col] = (data_fr[col] < lower) | (data_fr[col] > upper)

outliers_by_year = outlier_flags.groupby(data_fr["year"]).sum().sum(axis=1)

plt.figure(figsize=(8,5))
plt.bar(outliers_by_year.index, outliers_by_year.values)

plt.title("Total Outliers by Year")
plt.xlabel("Year")
plt.ylabel("Number of Outliers")

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

<h2 style="color: orange;">Section 4: Modelling & Analysis</h2>

In this section, I dive deep in analysing the data and looking for hidden patterns to find best model that describe happiness score of countries across the years from $2015$ to $2019$. It is divided based on model used as shown below.

In [ ]:
# droped useless columns + repeated columns)
drop_model_cols = ["happiness_score","happiness_rank","country","country_code",
    "Mobile ownership","Cybersecurity Index","school_enrol_tertiary_pct"]
data_fr = data_fr.drop(columns=["Region", "Cluster"], errors="ignore")

y = data_fr["happiness_score"].copy()
X = data_fr.drop(columns=drop_model_cols, errors="ignore").copy()
X = X.dropna(axis=1, how="all")
X = X.loc[:, X.nunique(dropna=True) > 1]

print( X.shape[1])
print(X.columns.tolist())

In [ ]:
# The reason I used this splitting rather than the typical splitting we tool in the course is that my data 
#is kinda repititive and same country may appear in both training and testing dataset cuz each country is mentioned
# 5 times (from 2015 to 2019) so my model might not have the opportunity to predict new (different from training subset)
# country. Thus, using he group shuffle split I will ensure that my models predict new countries that dont exists in
# the training sub set.
groups = data_fr["country_code"]

gss = GroupShuffleSplit(test_size=0.20, random_state=1810)
train_index, test_index = next(gss.split(X, y, groups=groups))

X_train_raw = X.iloc[train_index]
X_test_raw = X.iloc[test_index]
y_train = y.iloc[train_index]
y_test = y.iloc[test_index]
num_features = X_train_raw.select_dtypes(include=np.number).columns.tolist()
cat_features = X_train_raw.select_dtypes(exclude=np.number).columns.tolist()

In [ ]:
train_medians = X_train_raw[num_features].median()

X_train_num = X_train_raw[num_features].fillna(train_medians)
X_test_num = X_test_raw[num_features].fillna(train_medians)

X_train_cat = X_train_raw[cat_features].fillna("Missing")
X_test_cat = X_test_raw[cat_features].fillna("Missing")

X_train_cat, X_test_cat = X_train_cat.align(X_test_cat, join="left", axis=1, fill_value=0)

X_train = pd.concat([X_train_num, X_train_cat], axis=1)
X_test = pd.concat([X_test_num, X_test_cat], axis=1)

X_train = X_train.astype(float)
X_test = X_test.astype(float)

In [ ]:
# scaled variables cuz they sginificant differ (see Section 3)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame( scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test),columns=X_test.columns,index=X_test.index)

<h3 style="color: teal;">4.1 Multiple Linear Regression</h3>

In [ ]:
X_train_sm = sm.add_constant(X_train)
X_test_sm = sm.add_constant(X_test)

ols_model = sm.OLS(y_train, X_train_sm).fit()
print(ols_model.summary())

In [ ]:
y_train_pred_ols = ols_model.predict(X_train_sm)
y_test_pred_ols = ols_model.predict(X_test_sm)

In [ ]:
ols_residuals = y_test - y_test_pred_ols

plt.figure(figsize=(6,4))
plt.scatter(y_test_pred_ols, ols_residuals, alpha=0.7)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.title("OLS: Residuals vs Predicted")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(ols_residuals, bins=50)
plt.xlabel("Residuals")
plt.title("OLS: Residual Histogram")
plt.show()

In [ ]:
sm.qqplot(ols_residuals, line="45", fit=True)
plt.title("OLS: QQ Plot")
plt.show()

In [ ]:
y_train_pred_ols = ols_model.predict(X_train_sm)
y_test_pred_ols = ols_model.predict(X_test_sm)

train_rmse_ols = np.sqrt(mean_squared_error(y_train, y_train_pred_ols))
test_rmse_ols = np.sqrt(mean_squared_error(y_test, y_test_pred_ols))

train_r2_ols = r2_score(y_train, y_train_pred_ols)
test_r2_ols = r2_score(y_test, y_test_pred_ols)

print("OLS Train RMSE:", train_rmse_ols)
print("OLS Test RMSE :", test_rmse_ols)
print("OLS Train R2  :", train_r2_ols)
print("OLS Test R2   :", test_r2_ols)

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(y_test, y_test_pred_ols, alpha=0.7)

plt.plot([y_test.min(), y_test.max()],[y_test.min(), y_test.max()],color="red",linestyle="--")

plt.xlabel("Actual Happiness Score")
plt.ylabel("Predicted Happiness Score")
plt.title("OLS: Actual vs Predicted Values")
plt.grid(alpha=0.3)
plt.show()

<h3 style="color: teal;">4.2 Lasso Regression</h3>

In [ ]:
lasso_grid = GridSearchCV(Lasso(max_iter=100000),
    param_grid={"alpha": [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 5, 10]},
    cv=5,scoring="neg_mean_squared_error")

lasso_grid.fit(X_train_scaled, y_train)

lasso_model = lasso_grid.best_estimator_

print("Best alpha:", lasso_grid.best_params_)
print("Cross Validation RMSE:", np.sqrt(-lasso_grid.best_score_))

In [ ]:
y_train_pred_lasso = lasso_model.predict(X_train_scaled)
y_test_pred_lasso = lasso_model.predict(X_test_scaled)

train_rmse_lasso = np.sqrt(mean_squared_error(y_train, y_train_pred_lasso))
test_rmse_lasso = np.sqrt(mean_squared_error(y_test, y_test_pred_lasso))

train_r2_lasso = r2_score(y_train, y_train_pred_lasso)
test_r2_lasso = r2_score(y_test, y_test_pred_lasso)

print("Lasso Train RMSE:", train_rmse_lasso)
print("Lasso Test RMSE :", test_rmse_lasso)
print("Lasso Train R2  :", train_r2_lasso)
print("Lasso Test R2   :", test_r2_lasso)

In [ ]:
lasso_coef = pd.DataFrame({"variable": X_train_scaled.columns,"coefficient": lasso_model.coef_})

lasso_coef["absolute_coefficient"] = lasso_coef["coefficient"].abs()

lasso_selected = lasso_coef[lasso_coef["coefficient"] != 0]
lasso_selected = lasso_selected.sort_values("absolute_coefficient", ascending=False)

display(lasso_selected)

In [ ]:
print("Number of selected variables:", len(lasso_selected))
print("Number of removed variables:", len(lasso_coef) - len(lasso_selected))

In [ ]:
lasso_residuals = y_test - y_test_pred_lasso

plt.figure(figsize=(6,4))
plt.scatter(y_test_pred_lasso, lasso_residuals, alpha=0.7)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.title("Lasso: Residuals vs Predicted")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(y_test, y_test_pred_lasso, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color="red", linestyle="--")
plt.xlabel("Actual Happiness Score")
plt.ylabel("Predicted Happiness Score")
plt.title("Lasso: Actual vs Predicted")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(lasso_residuals, bins=40)
plt.xlabel("Residuals")
plt.ylabel("Frequency")
plt.title("Lasso: Residual Histogram")
plt.grid(axis="y", alpha=0.3)
plt.show()

<h3 style="color: teal;">4.3 Ridge Regression</h3>

In [ ]:
ridge_grid = GridSearchCV(Ridge(),
    param_grid={"alpha": [0.001, 0.01, 0.1, 1, 5, 10, 50, 100, 500]},
    cv=5,scoring="neg_mean_squared_error")

ridge_grid.fit(X_train_scaled, y_train)

ridge_model = ridge_grid.best_estimator_

print("Best alpha:", ridge_grid.best_params_)
print("Cross Validation RMSE:", np.sqrt(-ridge_grid.best_score_))

In [ ]:
y_train_pred_ridge = ridge_model.predict(X_train_scaled)
y_test_pred_ridge = ridge_model.predict(X_test_scaled)

train_rmse_ridge = np.sqrt(mean_squared_error(y_train, y_train_pred_ridge))
test_rmse_ridge = np.sqrt(mean_squared_error(y_test, y_test_pred_ridge))

train_r2_ridge = r2_score(y_train, y_train_pred_ridge)
test_r2_ridge = r2_score(y_test, y_test_pred_ridge)

print("Ridge Train RMSE:", train_rmse_ridge)
print("Ridge Test RMSE :", test_rmse_ridge)
print("Ridge Train R2  :", train_r2_ridge)
print("Ridge Test R2   :", test_r2_ridge)

In [ ]:
ridge_coef = pd.DataFrame({"variable": X_train_scaled.columns,"coefficient": ridge_model.coef_})

ridge_coef["absolute_coefficient"] = ridge_coef["coefficient"].abs()

ridge_coef = ridge_coef.sort_values("absolute_coefficient", ascending=False)
display(ridge_coef)

In [ ]:
ridge_top25 = ridge_coef.head(25).sort_values("absolute_coefficient")

plt.figure(figsize=(10, 8))
plt.barh(ridge_top25["variable"], ridge_top25["coefficient"])
plt.title("Top 25 Ridge Regression Coefficients")
plt.xlabel("Coefficient")
plt.ylabel("Variable")
plt.grid(axis="x", alpha=0.3)
plt.show()

In [ ]:
ridge_residuals = y_test - y_test_pred_ridge

plt.figure(figsize=(6,4))
plt.scatter(y_test_pred_ridge, ridge_residuals, alpha=0.7)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.title("Ridge: Residuals vs Predicted")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(ridge_residuals, bins=30)
plt.xlabel("Residuals")
plt.ylabel("Frequency")
plt.title("Ridge: Residual Histogram")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
sm.qqplot(ridge_residuals, line="45", fit=True)
plt.title("Ridge: QQ Plot")
plt.show()

<h3 style="color: teal;">4.4 Elastic Net Regression</h3>

In [ ]:
elastic_grid = GridSearchCV(ElasticNet(max_iter=100000),
    param_grid={"alpha": [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 5, 10],
        "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]},cv=5,scoring="neg_mean_squared_error")

elastic_grid.fit(X_train_scaled, y_train)
elastic_model = elastic_grid.best_estimator_

print("Best parameters:", elastic_grid.best_params_)
print("Cross Validation RMSE:", np.sqrt(-elastic_grid.best_score_))

In [ ]:
y_train_pred_elastic = elastic_model.predict(X_train_scaled)
y_test_pred_elastic = elastic_model.predict(X_test_scaled)

train_rmse_elastic = np.sqrt(mean_squared_error(y_train, y_train_pred_elastic))
test_rmse_elastic = np.sqrt(mean_squared_error(y_test, y_test_pred_elastic))

train_r2_elastic = r2_score(y_train, y_train_pred_elastic)
test_r2_elastic = r2_score(y_test, y_test_pred_elastic)

print("Elastic Net Train RMSE:", train_rmse_elastic)
print("Elastic Net Test RMSE :", test_rmse_elastic)
print("Elastic Net Train R2  :", train_r2_elastic)
print("Elastic Net Test R2   :", test_r2_elastic)

In [ ]:
elastic_coef = pd.DataFrame({"variable": X_train_scaled.columns,"coefficient": elastic_model.coef_})

elastic_coef["absolute_coefficient"] = elastic_coef["coefficient"].abs()

elastic_coef = elastic_coef.sort_values("absolute_coefficient", ascending=False)

In [ ]:
elastic_selected = elastic_coef[elastic_coef["coefficient"] != 0]
display(elastic_selected)

In [ ]:
print("Number of selected variables:", len(elastic_selected))
print("Number of removed variables:", len(elastic_coef) - len(elastic_selected))

In [ ]:
elastic_residuals = y_test - y_test_pred_elastic

plt.figure(figsize=(6,4))
plt.scatter(y_test_pred_elastic, elastic_residuals, alpha=0.7)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.title("Elastic Net: Residuals vs Predicted")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(elastic_residuals, bins=30)
plt.xlabel("Residuals")
plt.ylabel("Frequency")
plt.title("Elastic Net: Residual Histogram")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
sm.qqplot(elastic_residuals, line="45", fit=True)
plt.title("Elastic Net: QQ Plot")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(y_test, y_test_pred_elastic, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color="red", linestyle="--")

plt.xlabel("Actual Happiness Score")
plt.ylabel("Predicted Happiness Score")
plt.title("Elastic Net: Actual vs Predicted")
plt.grid(alpha=0.3)
plt.show()

<h3 style="color: teal;">4.5 Random Forests</h3>

In [ ]:
rf_grid = GridSearchCV(RandomForestRegressor(random_state=1810, n_jobs=-1),
    param_grid={"n_estimators": [100],"max_depth": [5, 10, None],"min_samples_leaf": [2, 4]},
    cv=3,scoring="neg_mean_squared_error", n_jobs=-1)

rf_grid.fit(X_train, y_train)

rf_model = rf_grid.best_estimator_

print("Best parameters:", rf_grid.best_params_)
print("Cross Validation RMSE:", np.sqrt(-rf_grid.best_score_))

In [ ]:
y_train_pred_rf = rf_model.predict(X_train)
y_test_pred_rf = rf_model.predict(X_test)

train_rmse_rf = np.sqrt(mean_squared_error(y_train, y_train_pred_rf))
test_rmse_rf = np.sqrt(mean_squared_error(y_test, y_test_pred_rf))

train_r2_rf = r2_score(y_train, y_train_pred_rf)
test_r2_rf = r2_score(y_test, y_test_pred_rf)

print("Random Forest Train RMSE:", train_rmse_rf)
print("Random Forest Test RMSE :", test_rmse_rf)
print("Random Forest Train R2  :", train_r2_rf)
print("Random Forest Test R2   :", test_r2_rf)

In [ ]:
rf_importance = pd.DataFrame({"variable": X_train.columns,"importance": rf_model.feature_importances_})

rf_importance = rf_importance.sort_values("importance", ascending=False)
display(rf_importance)

In [ ]:
rf_top25 = rf_importance.head(25).sort_values("importance")

plt.figure(figsize=(10, 8))
plt.barh(rf_top25["variable"], rf_top25["importance"])

plt.title("Top 25 Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Variable")
plt.grid(axis="x", alpha=0.3)
plt.show()

In [ ]:
rf_residuals = y_test - y_test_pred_rf

plt.figure(figsize=(6,4))
plt.scatter(y_test_pred_rf, rf_residuals, alpha=0.7)
plt.axhline(0, color="red", linestyle="--")

plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.title("Random Forest: Residuals vs Predicted")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(y_test, y_test_pred_rf, alpha=0.7)
plt.plot([y_test.min(), y_test.max()],[y_test.min(), y_test.max()],color="red",linestyle="--")
plt.xlabel("Actual Happiness Score")
plt.ylabel("Predicted Happiness Score")
plt.title("Random Forest: Actual vs Predicted")
plt.grid(alpha=0.3)
plt.show()

<h3 style="color: teal;">4.6 Gradient Boosting</h3>

In [ ]:
gb_grid = GridSearchCV(GradientBoostingRegressor(random_state=1810),
    param_grid={"n_estimators": [200],"learning_rate": [0.1],
        "max_depth": [2, 3],"min_samples_leaf": [2,4]},cv=3,
    scoring="neg_mean_squared_error")

gb_grid.fit(X_train, y_train)
gb_model = gb_grid.best_estimator_

print("Best parameters:", gb_grid.best_params_)
print("Cross Validation RMSE:", np.sqrt(-gb_grid.best_score_))

In [ ]:
y_train_pred_gb = gb_model.predict(X_train)
y_test_pred_gb = gb_model.predict(X_test)

train_rmse_gb = np.sqrt(mean_squared_error(y_train, y_train_pred_gb))
test_rmse_gb = np.sqrt(mean_squared_error(y_test, y_test_pred_gb))

train_r2_gb = r2_score(y_train, y_train_pred_gb)
test_r2_gb = r2_score(y_test, y_test_pred_gb)

print("Gradient Boosting Train RMSE:", train_rmse_gb)
print("Gradient Boosting Test RMSE :", test_rmse_gb)
print("Gradient Boosting Train R2  :", train_r2_gb)
print("Gradient Boosting Test R2   :", test_r2_gb)

In [ ]:
gb_importance = pd.DataFrame({"variable": X_train.columns,"importance": gb_model.feature_importances_})

gb_importance = gb_importance.sort_values("importance", ascending=False)

display(gb_importance)

In [ ]:
gb_top25 = gb_importance.head(25).sort_values("importance")

plt.figure(figsize=(10, 8))
plt.barh(gb_top25["variable"], gb_top25["importance"])

plt.title("Top 25 Gradient Boosting Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Variable")
plt.grid(axis="x", alpha=0.3)
plt.show()

In [ ]:
gb_residuals = y_test - y_test_pred_gb

plt.figure(figsize=(6,4))
plt.scatter(y_test_pred_gb, gb_residuals, alpha=0.7)
plt.axhline(0, color="red", linestyle="--")

plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.title("Gradient Boosting: Residuals vs Predicted")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(gb_residuals, bins=30)

plt.xlabel("Residuals")
plt.ylabel("Frequency")
plt.title("Gradient Boosting: Residual Histogram")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(y_test, y_test_pred_gb, alpha=0.7)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="red",
    linestyle="--"
)

plt.xlabel("Actual Happiness Score")
plt.ylabel("Predicted Happiness Score")
plt.title("Gradient Boosting: Actual vs Predicted")
plt.grid(alpha=0.3)
plt.show()

<h3 style="color: teal;">4.7 PCA + Regression</h3>

In [ ]:
pca = PCA(n_components=0.95)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Original number of predictors:", X_train_scaled.shape[1])
print("Number of PCA components:", pca.n_components_)
print("Explained variance:", pca.explained_variance_ratio_.sum())

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker="o")

plt.title("Cumulative Explained Variance by PCA Components")
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")

plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
pca_model = LinearRegression()

pca_model.fit(X_train_pca, y_train)

y_train_pred_pca = pca_model.predict(X_train_pca)
y_test_pred_pca = pca_model.predict(X_test_pca)

In [ ]:
train_rmse_pca = np.sqrt(mean_squared_error(y_train, y_train_pred_pca))
test_rmse_pca = np.sqrt(mean_squared_error(y_test, y_test_pred_pca))

train_r2_pca = r2_score(y_train, y_train_pred_pca)
test_r2_pca = r2_score(y_test, y_test_pred_pca)

print("PCA Train RMSE:", train_rmse_pca)
print("PCA Test RMSE :", test_rmse_pca)
print("PCA Train R2  :", train_r2_pca)
print("PCA Test R2   :", test_r2_pca)

In [ ]:
pca_residuals = y_test - y_test_pred_pca

plt.figure(figsize=(6,4))
plt.scatter(y_test_pred_pca, pca_residuals, alpha=0.7)
plt.axhline(0, color="red", linestyle="--")

plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.title("PCA Regression: Residuals vs Predicted")

plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(y_test, y_test_pred_pca, alpha=0.7)

plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color="red", linestyle="--")
plt.xlabel("Actual Happiness Score")
plt.ylabel("Predicted Happiness Score")
plt.title("PCA Regression: Actual vs Predicted")

plt.grid(alpha=0.3)
plt.show()

<h3 style="color: teal;">4.8 Summary</h3>

In [ ]:
model_results = pd.DataFrame({
    "Model": ["OLS","Lasso","Ridge","Elastic Net","Random Forest","Gradient Boosting","PCA Regression"],
    "Train RMSE": [train_rmse_ols,train_rmse_lasso,train_rmse_ridge,train_rmse_elastic,train_rmse_rf,train_rmse_gb,train_rmse_pca],
    "Test RMSE": [test_rmse_ols,test_rmse_lasso,test_rmse_ridge,test_rmse_elastic,test_rmse_rf,test_rmse_gb,test_rmse_pca],
    "Train R2": [train_r2_ols,train_r2_lasso,train_r2_ridge,train_r2_elastic,train_r2_rf,train_r2_gb,train_r2_pca],
    "Test R2": [test_r2_ols, test_r2_lasso, test_r2_ridge, test_r2_elastic, test_r2_rf, test_r2_gb, test_r2_pca]})

model_results = model_results.sort_values("Test RMSE")
display(model_results)

In [ ]:
plot_results = model_results.sort_values("Test RMSE", ascending=False)

plot_data = plot_results.melt(id_vars="Model",value_vars=["Train RMSE", "Test RMSE"],var_name="RMSE Type",value_name="RMSE")

plt.figure(figsize=(8,5))
sns.barplot(data=plot_data, x="RMSE", y="Model", hue="RMSE Type")
plt.title("Model Comparison by Train and Test RMSE")
plt.xlabel("RMSE")
plt.ylabel("Model")
plt.grid(axis="x", alpha=0.3)
plt.show()

Lasso performs best overall, with the lowest Test RMSE at $0.643$ and highest Test $R^2 =0.769$. Ridge and Elastic Net are very close. Gradient Boosting has very high Train $R^2 =0.950$ but lower Test $R^2 =0.761$, so it overfitted the data. OLS and Random Forest perform weaker on the test data.

<h3 style="color: teal;">4.9 Hypothesis Testing</h3>

In this sub section I am trying to have an statistical evidance if digital access variables, or education variables add a statistically meaningful improvement in explaining happiness score, generally speaking. so basically it answers the question of whether digital access and education are statistically related to happiness score of a country.

The hypotheses are of this sub section are:

Digital access test:

H0: Adding digital access variables does not improve the prediction of happiness score after controlling for year.

H1: Adding digital access variables improves the prediction of happiness score after controlling for year.

Education test:

H0: Adding education variables does not improve the prediction of happiness score after controlling for year.

H1: Adding education variables improves the prediction of happiness score after controlling for year.

In [ ]:
control_vars = ["year","Region_Europe & Central Asia","Region_Latin America & Caribbean","Region_Middle East & North Africa",
    "Region_Missing","Region_North America","Region_South Asia","Region_Sub-Saharan Africa","Cluster_Discoverer",
    "Cluster_Emerging","Cluster_Leader","Cluster_Transitioner"]

digital_vars = ["Internet_Use_Share","Index","Mobile Ownership","Local Relevance","Device affordability for poorest 40%",
    "Language accessibility of top ranked apps"]

education_vars = ["Tertiary_Enrollment_Gross","gov_exp_pct_gdp","lit_rate_adult_pct","pri_comp_rate_pct","pupil_teacher_primary",
    "pupil_teacher_secondary","school_enrol_primary_pct","school_enrol_secondary_pct"]

control_vars = [col for col in control_vars if col in X_train.columns]
digital_vars = [col for col in digital_vars if col in X_train.columns]
education_vars = [col for col in education_vars if col in X_train.columns]

In [ ]:
X_control = sm.add_constant(X_train[control_vars], has_constant="add")
control_model = sm.OLS(y_train, X_control).fit()

In [ ]:
X_digital = sm.add_constant(X_train[control_vars + digital_vars], has_constant="add")
digital_model = sm.OLS(y_train, X_digital).fit()

In [ ]:
X_education = sm.add_constant(X_train[control_vars + education_vars], has_constant="add")
education_model = sm.OLS(y_train, X_education).fit()

In [ ]:
X_full_group = sm.add_constant(X_train[control_vars + digital_vars + education_vars],has_constant="add")
full_group_model = sm.OLS(y_train, X_full_group).fit()

In [ ]:
digital_test = sm.stats.anova_lm(control_model, digital_model)
education_test = sm.stats.anova_lm(control_model, education_model)

digital_after_education_test = sm.stats.anova_lm(education_model, full_group_model)
education_after_digital_test = sm.stats.anova_lm(digital_model, full_group_model)

print("Controls vs Controls + Digital")
print("F-statistic:", digital_test.iloc[1]["F"])
print("p-value:", digital_test.iloc[1]["Pr(>F)"])
print()

print("Controls vs Controls + Education")
print("F-statistic:", education_test.iloc[1]["F"])
print("p-value:", education_test.iloc[1]["Pr(>F)"])
print()

print("Education model vs Education + Digital")
print("F-statistic:", digital_after_education_test.iloc[1]["F"])
print("p-value:", digital_after_education_test.iloc[1]["Pr(>F)"])
print()

print("Digital model vs Digital + Education")
print("F-statistic:", education_after_digital_test.iloc[1]["F"])
print("p-value:", education_after_digital_test.iloc[1]["Pr(>F)"])

In [ ]:
print("Controls only")
print("R2:", control_model.rsquared)
print("Adjusted R2:", control_model.rsquared_adj)
print("AIC:", control_model.aic)
print("BIC:", control_model.bic)
print()

print("Controls + Digital")
print("R2:", digital_model.rsquared)
print("Adjusted R2:", digital_model.rsquared_adj)
print("AIC:", digital_model.aic)
print("BIC:", digital_model.bic)
print()

print("Controls + Education")
print("R2:", education_model.rsquared)
print("Adjusted R2:", education_model.rsquared_adj)
print("AIC:", education_model.aic)
print("BIC:", education_model.bic)
print()

print("Controls + Digital + Education")
print("R2:", full_group_model.rsquared)
print("Adjusted R2:", full_group_model.rsquared_adj)
print("AIC:", full_group_model.aic)
print("BIC:", full_group_model.bic)

The hypothesis tests show that both groups significantly improve the model. Adding digital variables gave a very large F-statistic of $226.59$ with p-value = $2.55e^{-151}$ , while adding education variables gave F = $42.09$ with p-value = $1.61e^{-53}$. Digital access had the stronger effect because its $R^2$ increased to $0.691$, compared with $0.358$ for education. The full model had the highest $R^2 = 0.703$ and the lowest AIC = $1125.75$, so it fits best overall. However, the digital model had the lowest BIC = $1169.25$, meaning it is the simpler and more efficient model comapred to the full modell.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

model_steps = pd.DataFrame({
    "model": [
        "Controls only",
        "Controls + Education",
        "Controls + Digital",
        "Full model"
    ],
    "R2": [
        0.0000267,
        0.3576,
        0.6913,
        0.7032
    ],
    "AIC": [
        1844.815,
        1588.675,
        1133.877,
        1125.749
    ],
    "BIC": [
        1853.658,
        1632.891,
        1169.250,
        1196.495
    ]
})

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman"]

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(model_steps))

ax.plot(x, model_steps["R2"], marker="o", linewidth=2.5, markersize=9)

for i in range(len(model_steps)):
    ax.text(
        x[i],
        model_steps["R2"][i] + 0.025,
        f"$R^2$ = {model_steps['R2'][i]:.3f}",
        ha="center",
        fontsize=11
    )



ax.set_xticks(x)
ax.set_xticklabels(model_steps["model"], fontsize=11)
ax.set_ylabel("$R^2$", fontsize=13)
ax.set_xlabel("Model specification", fontsize=13)
ax.set_title("Incremental Contribution of Education and Digital Access to Happiness Score", fontsize=14, pad=15)

ax.set_ylim(0, 0.85)
ax.grid(axis="y", alpha=0.3)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig("../figures/model-comparison.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

coef_df = pd.DataFrame({
    "variable": X_train.columns,
    "coefficient": lasso_model.coef_
})

selected_df = coef_df[coef_df["coefficient"] != 0].copy()
selected_df["abs_coefficient"] = selected_df["coefficient"].abs()
selected_df = selected_df.sort_values("coefficient")

print("Number of predictors selected by Lasso:", len(selected_df))
print(selected_df[["variable", "coefficient"]])

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman"]

fig, ax = plt.subplots(figsize=(16, 14))

colors = ["#b2182b" if x < 0 else "#2166ac" for x in selected_df["coefficient"]]

ax.hlines(
    y=selected_df["variable"],
    xmin=0,
    xmax=selected_df["coefficient"],
    color=colors,
    linewidth=3,
    alpha=0.8
)

ax.scatter(
    selected_df["coefficient"],
    selected_df["variable"],
    s=160,
    color=colors,
    edgecolor="black",
    linewidth=0.8,
    zorder=3
)

ax.axvline(0, color="black", linewidth=1.2)

for _, row in selected_df.iterrows():
    if row["coefficient"] > 0:
        ax.text(
            row["coefficient"] + 0.005,
            row["variable"],
            f"{row['coefficient']:.3f}",
            va="center",
            ha="left",
            fontsize=14
        )
    else:
        ax.text(
            row["coefficient"] - 0.005,
            row["variable"],
            f"{row['coefficient']:.3f}",
            va="center",
            ha="right",
            fontsize=14
        )

ax.set_title("Predictors Selected by the Lasso Model", fontsize=20, pad=16)
ax.set_xlabel("Lasso Coefficient", fontsize=18)
ax.set_ylabel("Predictor", fontsize=18)

ax.grid(axis="x", alpha=0.25)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

ax.tick_params(axis="x", labelsize=14)
ax.tick_params(axis="y", labelsize=14)

plt.tight_layout()
plt.tight_layout()
plt.savefig("../figures/lasso-selected-predictors.png", dpi=600, bbox_inches="tight")
plt.show()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

coef_df = pd.DataFrame({
    "variable": X_train.columns,
    "coefficient": lasso_model.coef_
})

coef_df = coef_df[coef_df["coefficient"] != 0].copy()
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

print("Number of predictors selected by Lasso:", len(coef_df))

plot_df = coef_df.sort_values("abs_coefficient", ascending=False).head(20)
plot_df = plot_df.sort_values("coefficient")

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman"]

fig, ax = plt.subplots(figsize=(13, 8))

colors = ["#b2182b" if x < 0 else "#2166ac" for x in plot_df["coefficient"]]

ax.hlines(
    y=plot_df["variable"],
    xmin=0,
    xmax=plot_df["coefficient"],
    color=colors,
    linewidth=3,
    alpha=0.8
)

ax.scatter(
    plot_df["coefficient"],
    plot_df["variable"],
    s=170,
    color=colors,
    edgecolor="black",
    linewidth=0.8,
    zorder=3
)

ax.axvline(0, color="black", linewidth=1.2)

for _, row in plot_df.iterrows():
    if row["coefficient"] > 0:
        ax.text(row["coefficient"] + 0.005, row["variable"], f"{row['coefficient']:.3f}",
                va="center", ha="left", fontsize=10)
    else:
        ax.text(row["coefficient"] - 0.005, row["variable"], f"{row['coefficient']:.3f}",
                va="center", ha="right", fontsize=10)

ax.set_title("Predictors Selected by the Lasso Model", fontsize=16, pad=16)
ax.set_xlabel("Lasso Coefficient", fontsize=13)
ax.set_ylabel("Predictor", fontsize=13)

ax.grid(axis="x", alpha=0.25)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

ax.tick_params(axis="x", labelsize=10)
ax.tick_params(axis="y", labelsize=10)

plt.tight_layout()
plt.show()